# Qrisp Quantum Hackathon: Block Encoding Mastery

Welcome to the Qrisp Hackathon! Today you will learn how to use Qrisp's powerful `BlockEncoding` framework to bridge classical linear algebra with quantum computing, culminating in a quantum machine learning algorithm.

### Prerequisites
Run the cell below to set up your environment.

In [ ]:
# !pip install qrisp numpy
import os
os.environ["QRISP_SIMULATOR_FLOAT_THRESH"] = "1e-10"
os.environ["QRISP_SIMULATOR_CUTOFF_RATIO"] = "1e-10"
import networkx as nx
import numpy as np
from qrisp import QuantumBool, QuantumFloat, QuantumVariable, control, cx, h, prepare, swap, x
from qrisp.block_encodings import BlockEncoding
from qrisp.jasp import count_ops, terminal_sampling
from qrisp.operators import X, Y, Z
print("Environment initialized!")

---
## 🎯 Challenge 1: The Quantum Calculator (Matrix Arithmetic)

**The Scenario:**
You need to apply a composite matrix operator $C = 3A - B^2$ to a specific state vector $|x\rangle$. Instead of manipulating quantum circuits manually, Qrisp allows you to do this using NumPy-like syntax.

**The Task:**
1. Encode matrices $A$ and $B$ into block encodings.
2. Construct the operator $C$ using Qrisp's arithmetic overloads.
3. Apply operator $C$ to the state vector using the Repeat-Until-Success (`apply_rus`) protocol.
4. Compare with the classical result.

In [ ]:
A_mat = np.array([[ 0.5, 0.2], [ 0.2, 0.8]])
B_mat = np.array([[ 0.9, -0.1], [-0.1, 0.4]])
x_vec = np.array([1., 0.])

# 1. Create BlockEncodings for A and B
BE_A = # YOUR CODE HERE
BE_B = # YOUR CODE HERE

# 2. Compute C = 3A - B^2 using BlockEncoding arithmetic
BE_C = # YOUR CODE HERE

# Helper to prepare the state
def prep_x():
    qv = QuantumFloat(1)
    prepare(qv, x_vec)
    return qv

# 3. Apply the operator using Repeat-Until-Success
@terminal_sampling
def run_quantum_calculator():
    # YOUR CODE HERE
    return

res = run_quantum_calculator()
amps = np.sqrt([res.get(i, 0) for i in range(2)])
print("Quantum Result (Amplitudes):", amps)

In [ ]:
# 4. Compare with classical result
C_mat = # YOUR CODE HERE
res_vec = (C_mat @ x_vec) / np.linalg.norm(C_mat @ x_vec)
print("Is quantum result close to classical result:", np.allclose(amps, res_vec, atol=1e-3))

---
## 🎯 Challenge 2: Resource-Efficient Hamiltonians (LCU)

**The Scenario:**
You are simulating a 1D particle using a simplified shift operator. Your operator is $H = 2.0 I + 0.5 V + 0.5 V^\dagger$, where $V$ shifts a state $|k\rangle \rightarrow -|k+1 \pmod N\rangle$.

**The Task:**
1. Define the Python functions for the Identity, $V$, and $V^\dagger$ unitaries acting on a `QuantumVariable`.
2. Use `BlockEncoding.from_lcu` to construct the encoding.
3. Compare the gate counts and depth of your LCU method versus doing the same with `from_array`.

In [ ]:
H_mat = 2*np.eye(16)+0.5*np.eye(16, k=1) + 0.5*np.eye(16, k=-1)
H_mat[0, 15] = 0.5
H_mat[15, 0] = 0.5
print(H_mat)

In [ ]:
# 1. Define the unitaries
def I_op(qv):
    pass # Identity does nothing

def V_op(qv):
    # Shift forward
    qv += 1

def V_dg_op(qv):
    # YOUR CODE HERE (Shift backward)
    pass

# 2. Construct the BlockEncoding from LCU
unitaries = [I_op, V_op, V_dg_op]
coeffs = np.array([2.0, 0.5, 0.5])

BE_H_lcu = # YOUR CODE HERE

# 3. Analyze Resources
qf = QuantumFloat(4)
print("LCU Resources:\n", BE_H_lcu.resources(qf))

In [ ]:
# 4. Construct the BlockEncoding for H from its array
BE_H_array = # YOUR CODE HERE

# 5. Analyze Resources for the array-based BlockEncoding
qf = QuantumFloat(4)
print("Array-based Resources:\n", BE_H_array.resources(qf))

---
## 🎯 Challenge 3: Quantum Linear Systems Problem

**The Scenario:**
You are solving the system $Ax = b$. In traditional quantum computing, you'd use the HHL algorithm, which is incredibly difficult to build from scratch. Qrisp abstracts this via spectral transformations.

**The Task:**
Use the `.inv()` method on a BlockEncoding to apply the inverse of $A$ to the vector $b$. Calculate the classical NumPy inverse for comparison.

In [ ]:
A_mat = np.array([[ 0.7, -0.2, 0.0, 0.0],
                  [-0.2,  0.8, -0.1, 0.0],
                  [ 0.0, -0.1,  0.9, -0.2],
                  [ 0.0,  0.0, -0.2,  0.6]])
b_vec = np.array([0.5, 0.5, 0.5, 0.5])

epsilon = 0.01
kappa = np.linalg.cond(A_mat)

BE_A_sys = BlockEncoding.from_array(A_mat)

# 1. Create the block encoding of the inverse matrix
BE_A_inv = # YOUR CODE HERE

def prep_b():
    qv = QuantumFloat(2)
    prepare(qv, b_vec)
    return qv

@terminal_sampling
def solve_qlsp():
    # 2. Apply the block encoding of the inverted matrix to prep_b using RUS
    return # YOUR CODE HERE

res = solve_qlsp()
amps = np.sqrt([res.get(i, 0) for i in range(4)])
print("Quantum Solution:", amps)

# 3. Classical check (Normalize the classical vector for 1:1 comparison)
# YOUR CODE HERE

---
## 💀 Final Boss: The Quantum Recommender System

**The Scenario: Filtering the Similarity Graph**
You are building a quantum recommendation engine. Instead of a User-Item matrix, you are working with an **Item-Item Similarity Matrix** $A$. Both rows and columns represent movies, and the values represent how correlated they are (hence, the matrix is symmetric/Hermitian).

The matrix contains strong "core genres" (dominant eigenvalues) but also weak, spurious correlations (small eigenvalues) from random viewer behavior. To generate a high-quality recommendation for a user (state vector $|x\rangle$), you want to project their watch history through a non-linear spectral filter to crush the noise.

**The Task:**
1. You are given a Hermitian similarity matrix $A$ and a new user's viewing history vector $x$.
2. Define a polynomial filter $P(x) = x^3$ to act as an extreme contrast filter on the matrix spectrum.
3. Because the matrix is Hermitian, you can use Qrisp's `.poly()` (or `.svt()`) method to apply this polynomial directly to the block-encoded operator.
4. Apply the filtered operator to the user's vector to get their denoised recommendation profile!

In [ ]:
# A Hermitian Item-Item Similarity Matrix (Movies x Movies)
A_data = np.array([[ 0.8,  0.4, -0.2,  0.1],
                   [ 0.4,  0.7,  0.1,  0.0],
                   [-0.2,  0.1,  0.9,  0.3],
                   [ 0.1,  0.0,  0.3,  0.6]])

# The user loved Movie 1, liked Movie 3, and has not seen 2 or 4.
x_vec = np.array([1.0, 0.0, 0.8, 0.0])

# 1. Create the BlockEncoding for A_data
BE_A_data = # YOUR CODE HERE

# 2. Define the polynomial coefficients for P(x) = x^3
poly_coeffs = # YOUR CODE HERE

# 3. Apply the Polynomial Transformation
BE_filtered = # YOUR CODE HERE

def prep_x_recs():
    qv = QuantumFloat(2)
    prepare(qv, x_vec)
    return qv

@terminal_sampling
def run_quantum_recommender():
    # 4. Apply the filtered operator using RUS
    return # YOUR CODE HERE

res = run_quantum_recommender()
amps = np.sqrt([res.get(i, 0) for i in range(4)])
print("Denoised User Profile (Quantum):", amps)

# --- Classical Check ---
# For a Hermitian matrix, applying P(x) = x^3 to the eigenvalues 
# is mathematically identical to just cubing the matrix! (A * A * A)
C_filtered = A_data @ A_data @ A_data
c_sol = C_filtered @ x_vec
c_sol_norm = c_sol / np.linalg.norm(c_sol)
print("Denoised User Profile (Classical): ", c_sol_norm)

---
## 💀 Bonus: Ground state preparation

Ground state preparation is a fundamental task in quantum computation, serving as a cornerstone for simulating strongly correlated systems in chemistry, material sciences and physics.

Understanding the ground state, i.e., the state of lowest energy, is essential for predicting a system's stability, electronic properties, and chemical reactivity.

However, **finding the ground state is a notoriously difficult challenge**. In fact, ground state preparation is QMA-complete, meaning it is generally hard even for quantum computers to find the exact solution for any arbitrary Hamiltonian. In practice, we often circumvent this complexity by starting with a 'good enough' initial guess (a trial state) and employing quantum algorithms to systematically lower the system's energy, a process often referred to as algorithmic cooling.

Let us consider an $L$-qubit antiferromagnetic Heisenberg model

$$ H = \sum_{i=0}^{L-1}(X_iX_{i+1} + 0.3 Y_iY_{i+1} + 0.5 Z_iZ_{i+1}) $$

Physically, [antiferromagnetism](https://en.wikipedia.org/wiki/Antiferromagnetism) describes materials where neighboring spins "want" to point in opposite directions $\uparrow\downarrow$ to reach the lowest energy state. Unlike a standard fridge magnet (where spins point the same way), an antiferromagnet has no net external magnetism. These models are crucial for understanding quantum materials, such as high-temperature superconductors.

As initial system state we choose a tensor product of Singlet states

$$ \ket{\text{Singlet}} = 2^{-L/4}(\ket{10}-\ket{01})^{\otimes L/2} $$

of consecutive qubits. Starting at our initial state $\ket{\text{Singlet}}$, we aim to prepare a state with an even higher overlap with the ground state of the operator $H$

**The Task:** Apply a quantum algorithm that prepares the gound state of the system with high fidelity, starting form the Singlet state. You may use **any algorithm in Qrisp**, except for ``prepare``.

In [ ]:
def generate_1D_chain_graph(L):
    graph = nx.Graph()
    graph.add_edges_from([(k, (k+1)%L) for k in range(L-1)])
    return graph


# Define Heisenberg Hamiltonian with spectrum in [-1,1].
L = 8
G = generate_1D_chain_graph(L)
H = sum((X(i) * X(j) + 0.3 * Y(i) * Y(j) + 0.5 * Z(i) * Z(j)) for i, j in G.edges())

# Normalizes the Hamiltonian so that the sum of the absolute values of its coefficients is 1.
# This ensurres that all eigenvlaues lie in [-1,1]
H = (1 / np.sum(np.abs(H.coeffs()))) * H


# Prepares the singlet state $\ket{10}-\ket{01} of two qubits
def sing(qv, i, j):
    x(qv[i])
    h(qv[i])
    x(qv[j])
    cx(qv[i], qv[j])

# Define initial state preparation function
# preparing a tensor product of Singlet states.
def psi_prep():
    operand = QuantumFloat(L)
    for i in range(0, L, 2):
        sing(operand, i, i+1)
    return operand

Let's analyse the spectrum of $H$ clasically!

While this classical analysis does not scale to larger systems, it provides crucial intuition for the filtering approach by illustrating how it acts upon the Hamiltonian spectrum.
Let us take a closer look at the spectrum of this Hamiltonian, and the inital state $\ket{\psi_0}=\ket{\text{Singlet}}$. 
First, we compute all eigenvalues and the corresponding eigenvectors. Secondly, we calculate the fidelities of the initial state $\ket{\psi_0}$ and each eigenstate $\ket{\lambda_k}$.
Next, we find the indices $k$ of the eigenstates $\ket{\lambda_k}$ that have a significant overlap with the initial state, i.e., $F_k=|\langle \lambda_k | \psi_0 \rangle|^2>0.01$.

In [ ]:
H_matrix = H.to_array()
eigvals, eigvecs = np.linalg.eigh(H_matrix)

idx = np.argsort(eigvals)
eigvals_sorted = eigvals[idx].real
eigvecs_sorted = eigvecs[:, idx]
print(f'Ground state energy: {eigvals_sorted[0]}\n') # lambda_0
print(f'First excited state energy: {eigvals_sorted[1]}\n') # lambda_1

psi0 = psi_prep().qs.statevector_array()
fidelities = np.abs(eigvecs_sorted.conj().T @ psi0) ** 2

threshold = 0.01
significant_mask = fidelities > threshold
indices = np.where(significant_mask)[0]

print(f'Eigenvalues: {eigvals_sorted[indices]}\n')
print(f'Fidelities: {fidelities[indices]}\n')

**Proposed approach:**

Starting at our initial state $\ket{\psi_0}$, we aim to prepare a state with an even higher overlap with the ground state $\ket{\lambda_0}$. 
Therefore, we aim to apply a filter, e.g., [Gaussian filter](https://en.wikipedia.org/wiki/Gaussian_filter) centered at $\lambda_0$, or a step function filter, which suppresses higher energy components: indeed for 

$$ H = \sum_i\lambda_i\ket{\lambda_i}\bra{\lambda_i}\quad \text{and}\quad \ket{\psi_0} = \sum_i\alpha_i\ket{\lambda_i} $$

the state $\ket{\psi}$ after applying the filter $f$ is

$$ \ket{\psi} = \frac{f(H)\ket{\psi_0}}{\|f(H)\ket{\psi_0}\|} $$

where

$$ f(H)\ket{\psi_0} = \left(\sum_i f(\lambda_i)\ket{\lambda_i}\bra{\lambda_i}\right) \sum_j\alpha_j\ket{\lambda_j} = \sum_i f(\lambda_i)\alpha_i\ket{\lambda_i} $$

Thus, the magnitude of the amplitude of $\ket{\lambda_i}$ for $i>0$ is suppressed relative to the amplitude of $\ket{\lambda_0}$ in the state $\ket{\psi}$.

This filter is approximated by [Chebyshev polynomials](https://en.wikipedia.org/wiki/Chebyshev_polynomials) of the first kind. 

In [ ]:
#from numpy.polynomial import Chebyshev
#from numpy.polynomial.chebyshev import chebval

# Chebyshev fit
# x_nodes = np.cos(np.pi * np.arange(201) / 200)
# f_nodes = your_function(x_nodes)
# cheb_fit = Chebyshev.fit(x_nodes, f_nodes, deg=100)
# cheb_coeffs = cheb_fit.coef

In [ ]:
# You may use any algorithm in Qrisp, except for 'prepare'.
# This function receives a QuantumFloat in tensor product of singlet states.
def state_preparation(operand):
    
    # YOUR CODE HERE
    # When using block encodings, write:
    # ancs = BE.apply(operand) 
    # YOUR CODE HERE

    # If your algorithm requires ancillae that must be measured 
    # in the zero state (e.g., when using block encodings), 
    # return a list of ancialla variables:
    # return ancs 
    return [QuantumBool()]

In [ ]:
# DO NOT MODIFY THIS CELL
# Run this cell to check your solution.
def check_solution(state_preparation):

    def wrapped_state_preparation():
        operand = psi_prep()
        ancs = state_preparation(operand)
        return operand, *ancs

    def prepare_ground_state():
        operand_ref = QuantumVariable(L)
        prepare(operand_ref, eigvecs[:, 0], reversed=True)
        return operand_ref

    @terminal_sampling
    def check_solution_helper(trial_function): 

        operand, *ancs = trial_function()
        operand_ref = prepare_ground_state()

        # Perform the swap test to measure the fidelity between the prepared state and the ground state
        flag = QuantumFloat(1)
        h(flag)
        with control(flag[0]):
            swap(operand, operand_ref)
        h(flag)

        return flag, *ancs


    gate_counts = count_ops(meas_behavior="0")(wrapped_state_preparation)()
    total_gate_count = sum(gate_counts.values())

    res_dict = check_solution_helper(wrapped_state_preparation)
    # Post-selection on ancillas being in |0> state
    filtered_dict = {k[0]: p for k, p in res_dict.items() \
                    if all(x == 0 for x in k[1:])}
    success_prob = sum(filtered_dict.values())
    filtered_dict = {k: p / success_prob for k, p in filtered_dict.items()}

    p0 = filtered_dict.get(0, 0)
    fidelity = 2 * p0 - 1
    if fidelity > 0.99:
        print("Congratulations! The solution is correct with very high fidelity:", fidelity)
    elif fidelity > 0.90:
        print("The solution is correct with high fidelity:", fidelity)
    else:
        print("The solution is not accurate enough. Fidelity:", fidelity)

    print("Success probability:", success_prob)
    print("Gate counts per repetition:", gate_counts)
    print("Total cost ((success probability) x (total gate count)):", success_prob * total_gate_count)


check_solution(state_preparation)